## Задание 1

In [1]:
!mkdir -p data
!wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip -P data
!unzip -o data/lenta.txt.zip -d data/

--2026-10-29 15:36:46--  https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip [following]
--2026-10-29 15:36:47--  https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5723675 (5.5M) [application/zip]
Saving to: ‘data/lenta.txt.zip’

lenta.txt.zip       100%[===================>]   5.46M  --.-KB/s    in 0.08s   

2026-10-29 15:36:47 (71.6 MB/s) - ‘data/lenta.txt.zip’ sa

In [16]:
import re
from collections import Counter

with open('data/lenta.txt', encoding='utf-8') as f:
    text = f.read()

# Слова с кириллицей и латиницей, включая внутренние дефисы
pattern = re.compile(
    r'(?<![\w-])'
    r'(?=[А-Яа-яЁёA-Za-z-]*[А-Яа-яЁё])'
    r'(?=[А-Яа-яЁёA-Za-z-]*[A-Za-z])'
    r'[А-Яа-яЁёA-Za-z]+(?:-[А-Яа-яЁёA-Za-z]+)*'
    r'(?![\w-])'
)

mixed = pattern.findall(text)
cnt = Counter(mixed)

print('найдено вхождений:', len(mixed))
print('уникальных слов:', len(cnt))
print('\nсамые частотные слова:')
for word, count in cnt.most_common(15):
    print(f'{word}: {count}')

print('\nответ:', cnt.most_common(1)[0][0])

найдено вхождений: 669
уникальных слов: 449

самые частотные слова:
Яndex: 38
web-сайтов: 8
web-страниц: 7
cообщает: 7
web-страницы: 7
Yтро: 6
СNN: 6
Cотрудники: 6
Аssociated: 6
web-странице: 6
DDoS-атак: 6
Cтатье: 6
web-сайта: 5
web-сайт: 5
cовета: 5

ответ: Яndex


## Задание 2

In [17]:
import string

PUNCT = string.punctuation + '«»—–…„“”‘’'

with open('data/lenta.txt', encoding='utf-8') as f:
    text = f.read().lower()

tokens = [t for t in (w.strip(PUNCT) for w in text.split()) if t]

seq = 'об этом сообщает агентство интерфакс со ссылкой на пресс-службу'.split()

tris = [tuple(seq[i:i+3]) for i in range(len(seq) - 2)]
need_tri = set(tris)
need_bi = {t[:2] for t in tris}

tri_cnt = Counter(
    t for t in zip(tokens, tokens[1:], tokens[2:])
    if t in need_tri
)

bi_cnt = Counter(
    b for b in zip(tokens, tokens[1:])
    if b in need_bi
)

results = []

for t in tris:
    c2 = bi_cnt[t[:2]]
    if c2 == 0:
        continue

    c3 = tri_cnt[t]
    p = c3 / c2
    results.append((p, ' '.join(t), c3, c2))

for p, s, c3, c2 in results:
    print(f'{p:.8f}  {s}  ({c3}/{c2})')

if results:
    pmin = min(r[0] for r in results)
    ties = [r[1] for r in results if r[0] == pmin]

    print('\nминимальная вероятность:', pmin)
    print('ответ:', ties[0])

    if len(ties) > 1:
        print('одинаковая минимальная вероятность у:', ties)
else:
    print('нет биграмм с ненулевой частотой')

0.15990991  об этом сообщает  (284/1776)
0.07638889  этом сообщает агентство  (22/288)
0.02034884  сообщает агентство интерфакс  (7/344)
0.07407407  агентство интерфакс со  (2/27)
0.96774194  интерфакс со ссылкой  (180/186)
0.99200639  со ссылкой на  (1241/1251)
0.11146752  ссылкой на пресс-службу  (139/1247)

минимальная вероятность: 0.020348837209302327
ответ: сообщает агентство интерфакс


## Задание 3

In [19]:
import pandas as pd

lenta = pd.read_csv("https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip").sample(500, random_state=42)

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer

sub = lenta.sample(500, random_state=8).reset_index(drop=True)
texts = sub['text'].astype(str).tolist()

model = SentenceTransformer('cointegrated/rubert-tiny2')
emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

sim = emb @ emb.T
np.fill_diagonal(sim, -np.inf)

for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        if texts[i] == texts[j]:
            sim[i, j] = sim[j, i] = -np.inf

i, j = np.unravel_index(np.argmax(sim), sim.shape)
print('cos =', sim[i, j])
print(texts[i][:200], '\n---\n', texts[j][:200])
print('ответ:', texts[i].split()[0], texts[j].split()[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

...

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

cos = 0.9411663
Объемы производства легковых автомобилей в России в июле текущего года выросли на 19,7 процента по отношению к июлю прошлого года. А за первые семь месяцев этот показатель вырос на 20,5 процента к ана 
---
 Объем рынка новых легковых автомобилей сегмента Luxury в России по итогам первых  шести месяцев 2017 года составил 733 единицы. Это на 9 процентов выше прошлогоднего показателя, когда было продано 675
ответ: Объемы Объем
